# Silver Layer – Standardized & Trusted Healthcare Data

The **Silver Layer** transforms raw Bronze ingestion into cleaned, deduplicated, and reliable datasets.  
This layer ensures that downstream business logic in the Gold layer operates on **consistent and trustworthy data**.

The Silver tables serve as the **operational backbone** for the three business use cases:

- Claim Decision (Adjudication)
- Fraud Detection
- Financial Reconciliation

---

# Use Case 1 – Claim Decision (Gold_Claim_Decision)

This use case determines whether a claim is approved based on policy coverage and total charges.

### Silver tables supporting this logic

### `claim`
- Deduplicates claim records using claim_id  
- Ensures one trusted claim header per claim  
- Used to identify patient, provider, policy, and claim date  

### `claim_line`
- Deduplicates procedures per claim  
- Ensures each medical service is counted once  
- Used to compute total claim charge  

### `policy`
- Deduplicates insurance policies  
- Ensures accurate coverage limits  
- Used to determine approved amount and decision status  

Together, these tables allow the Gold layer to calculate:
- Total claim cost
- Coverage eligibility
- Approval amount

---

# Use Case 2 – Fraud Detection (Gold_Fraud_Signals)

This use case evaluates whether a claim looks suspicious based on cost patterns and behaviour signals.

### Silver tables supporting this logic

### `claim`
- Provides the claim identity and provider/patient relationships  
- Used for linking claim to fraud signals  

### `claim_line`
- Ensures clean aggregation of medical charges  
- Used to compute total charge for fraud rules  

### `provider`
- Deduplicates provider records  
- Enables future provider-level fraud pattern checks  

### `patient`
- Deduplicates patient identity  
- Supports future repeat-visit or abnormal behavior checks  

These tables allow the Gold layer to:
- Detect high-cost claims
- Assign fraud severity
- Track fraud disposition workflow

---

# Use Case 3 – Financial Reconciliation (Gold_Reconciliation)

This use case tracks whether approved claims are actually paid and settled correctly.

### Silver tables supporting this logic

### `claim`
- Provides the list of all claims that require payment tracking  
- Ensures finance monitors every claim  

### `claim_line`
- Supplies total billed amount per claim  
- Represents what the provider requested  

### `payment`
- Deduplicates payment records  
- Ensures accurate payment tracking and timestamps  
- Used to compute paid amount and settlement status  

### `payer`
- Deduplicates insurance companies  
- Enables payer-level settlement reporting  

These tables allow the Gold layer to:
- Compare billed vs paid
- Track settlement progress
- Identify pending or underpaid claims
- Monitor payment timelines

---

# What Silver Transformations Are Doing Technically

Across all tables, Silver applies consistent rules:

### Watermarking
Used to handle late-arriving streaming data safely.

### Deduplication
Ensures only one trusted record per business key:
- provider_id
- payer_id
- patient_id
- policy_id
- claim_id
- procedure_code
- payment_id

This prevents duplicate financial or operational calculations in Gold.

### Trusted Delta Tables
Silver tables become the single standardized source for:
- analytics
- joins
- aggregations
- downstream business rules

---

# Business Value of the Silver Layer

-Removes duplicate claims, payments, and procedures  
-Guarantees accurate financial totals  
-Ensures fraud rules evaluate correct values  
-Provides consistent policy and provider mapping  
-Creates reliable foundation for Gold analytics  

---

The Silver layer ensures that **every decision, fraud signal, and financial metric in the Gold layer is built on trusted data.**


In [0]:
# %sql
# CREATE VOLUME IF NOT EXISTS healthcare.streaming.chk_silver;


In [0]:
bronze_db = "healthcare.stream_bronze"
silver_db = "healthcare.stream_silver"
chk_base = "/Volumes/healthcare/streaming/chk_silver/"


In [0]:
from pyspark.sql.functions import col

provider_stream = spark.readStream.table(f"{bronze_db}.provider")

provider_clean = (
    provider_stream
    .withWatermark("ingestion_ts", "1 day")
    .dropDuplicates(["provider_id"])
)

(
    provider_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "provider")
    .trigger(availableNow=True)
    .toTable(f"{silver_db}.provider")
)


In [0]:
payer_stream = spark.readStream.table(f"{bronze_db}.payer")

payer_clean = (
    payer_stream
    .withWatermark("ingestion_ts", "1 day")
    .dropDuplicates(["payer_id"])
)

(
    payer_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "payer")
    .trigger(availableNow=True)
    .toTable(f"{silver_db}.payer")
)


In [0]:
patient_stream = spark.readStream.table(f"{bronze_db}.patient")

patient_clean = (
    patient_stream
    .withWatermark("ingestion_ts", "1 day")
    .dropDuplicates(["patient_id"])
)

(
    patient_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "patient")
    .trigger(availableNow=True)
    .toTable(f"{silver_db}.patient")
)


In [0]:
policy_stream = spark.readStream.table(f"{bronze_db}.policy")

policy_clean = (
    policy_stream
    .withWatermark("ingestion_ts", "1 day")
    .dropDuplicates(["policy_id"])
)

(
    policy_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "policy")
    .trigger(availableNow=True)
    .toTable(f"{silver_db}.policy")
)


In [0]:
claim_stream = spark.readStream.table(f"{bronze_db}.claim")

claim_clean = (
    claim_stream
    .withWatermark("ingestion_ts", "1 day")
    .dropDuplicates(["claim_id"])
)

(
    claim_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "claim")
    .trigger(availableNow=True)
    .toTable(f"{silver_db}.claim")
)


In [0]:
claim_line_stream = spark.readStream.table(f"{bronze_db}.claim_line")

claim_line_clean = (
    claim_line_stream
    .withWatermark("ingestion_ts", "1 day")
    .dropDuplicates(["claim_id","procedure_code"])
)

(
    claim_line_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "claim_line")
    .trigger(availableNow=True)
    .toTable(f"{silver_db}.claim_line")
)


In [0]:
payment_stream = spark.readStream.table(f"{bronze_db}.payment")

payment_clean = (
    payment_stream
    .withWatermark("ingestion_ts", "1 day")
    .dropDuplicates(["payment_id"])
)

(
    payment_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_base + "payment")
    .trigger(availableNow=True)
    .toTable(f"{silver_db}.payment")
)


### Silver Streaming Completed

All Bronze tables were:
- deduplicated using watermark logic
- converted into clean Silver datasets
- ready for downstream adjudication, reconciliation, and fraud logic
